In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = "alxxtexxr/VRBI-Dataset"
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

In [4]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import time

# Load all batch file paths
batch_paths = [f for f in vision_data_dir.glob('*.npz')]

# Initialize MiniBatchKMeans
k = 16384 # Number of clusters
kmeans = MiniBatchKMeans(
    n_clusters=k,
    init='random',
    batch_size=16384,
    n_init=1,
    reassignment_ratio=0.01,
    max_no_improvement=10,
    random_state=42
)

# Train in batches
# batch_paths -> current: 10 files
for i, batch_path in enumerate(batch_paths):
    print("="*128)
    print(f"Processing batch {i+1}/{len(batch_paths)}: {batch_path}")
    print("="*128)
    
    start_time = time.time()
    
    batch = np.load(batch_path)
    visual_embs = batch['visual_embs'].astype(np.float32) # (N, 2048) -> current: (32_400, 2048)
    
    # L2 normalize per vector
    visual_embs_unit = normalize(visual_embs, axis=1)
    
    print("Normalized visual embeddings shape:", visual_embs_unit.shape)
    print("Mean L2 norm (should be ~1):", np.linalg.norm(visual_embs_unit, axis=1).mean())
    print()
    
    # Update k-means
    kmeans.partial_fit(visual_embs_unit)
    
    end_time = time.time()
    iter_time = end_time - start_time
    
    print(f"Time taken: {iter_time:.2f}s")
    print()

In [5]:
import joblib

codebook_unit = kmeans.cluster_centers_

kmeans_path = vision_data_dir / 'kmeans.pkl'
codebook_unit_path = vision_data_dir / 'codebook_unit.npy'

joblib.dump(kmeans, kmeans_path)
np.save(codebook_unit_path, codebook_unit)

print("K-means model saved to:", kmeans_path)
print("Normalized codebook saved to:", codebook_unit_path)

In [6]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj=str(kmeans_path),
    path_in_repo=str(kmeans_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=str(codebook_unit_path),
    path_in_repo=str(codebook_unit_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)